In [1]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("week8-assignment-6"). \
config("spark.sql.warehouse.dir", f"/user/itv024128/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()
from pyspark.sql.types import *
from pyspark.sql.functions import *

## log analysis usecase using spark

In [2]:
## /public/trendytech/datasets/logdata1m.csv    #12months #5logging levels

In [3]:
# INFO,2015-8-8 20:49:22
# WARN,2015-1-14 20:05:00
# INFO,2017-6-14 00:08:35
# INFO,2016-1-18 11:50:14
# DEBUG,2017-7-1 12:55:02

In [4]:
schema1 = 'loglevel string , logtime string'

In [5]:
logdf = spark.read.format("csv").schema(schema1).load("/public/trendytech/datasets/logdata1m.csv")

In [6]:
logdf.show(5)

+--------+------------------+
|loglevel|           logtime|
+--------+------------------+
|    INFO| 2015-8-8 20:49:22|
|    WARN|2015-1-14 20:05:00|
|    INFO|2017-6-14 00:08:35|
|    INFO|2016-1-18 11:50:14|
|   DEBUG| 2017-7-1 12:55:02|
+--------+------------------+
only showing top 5 rows



In [7]:
logdf1 = logdf.withColumn("logtime",to_timestamp("logtime"))  ## convert string to timestamp

In [8]:
logdf1.show(5)

+--------+-------------------+
|loglevel|            logtime|
+--------+-------------------+
|    INFO|2015-08-08 20:49:22|
|    WARN|2015-01-14 20:05:00|
|    INFO|2017-06-14 00:08:35|
|    INFO|2016-01-18 11:50:14|
|   DEBUG|2017-07-01 12:55:02|
+--------+-------------------+
only showing top 5 rows



In [9]:
logdf1.printSchema()

root
 |-- loglevel: string (nullable = true)
 |-- logtime: timestamp (nullable = true)



In [10]:
logdf2 = logdf1.select( "loglevel", date_format(col("logtime"),"MMMM").alias("month"))

In [11]:
logdf2.show(10)

+--------+--------+
|loglevel|   month|
+--------+--------+
|    INFO|  August|
|    WARN| January|
|    INFO|    June|
|    INFO| January|
|   DEBUG|    July|
|    INFO|February|
|    INFO|    July|
|    INFO|   April|
|   DEBUG|November|
|    INFO|  August|
+--------+--------+
only showing top 10 rows



In [12]:
logdf3 = logdf2.groupBy("loglevel","month").count().orderBy("loglevel")

In [13]:
logdf3.show(10)

+--------+--------+-----+
|loglevel|   month|count|
+--------+--------+-----+
|   DEBUG|   April|41869|
|   DEBUG|   March|41652|
|   DEBUG|February|41734|
|   DEBUG|November|33366|
|   DEBUG|    June|41774|
|   DEBUG| October|41936|
|   DEBUG|    July|42085|
|   DEBUG|     May|41785|
|   DEBUG| January|41961|
|   DEBUG|December|41749|
+--------+--------+-----+
only showing top 10 rows



In [36]:
logdf4 = logdf1.withColumn("month_num",month("logtime")).groupBy("loglevel","month_num").count().orderBy("month_num")

In [38]:
# or logdf4 = logdf1.withColumn("month_num",date_format(col("logtime"), "M").cast("int")).groupBy("loglevel","month_num").count().orderBy("month_num")

In [37]:
logdf4.show(20)

+--------+---------+-----+
|loglevel|month_num|count|
+--------+---------+-----+
|   FATAL|        1|   94|
|   DEBUG|        1|41961|
|    WARN|        1| 8217|
|    INFO|        1|29119|
|   ERROR|        1| 4054|
|   ERROR|        2| 4013|
|   FATAL|        2|   72|
|    INFO|        2|28983|
|    WARN|        2| 8266|
|   DEBUG|        2|41734|
|    WARN|        3| 8165|
|    INFO|        3|29095|
|   ERROR|        3| 4122|
|   DEBUG|        3|41652|
|   FATAL|        3|   70|
|    WARN|        4| 8277|
|   FATAL|        4|   83|
|   ERROR|        4| 4107|
|    INFO|        4|29302|
|   DEBUG|        4|41869|
+--------+---------+-----+
only showing top 20 rows



### SQL style

In [14]:
logdf1.createOrReplaceTempView("serverlogs")

In [15]:
spark.sql("select * from serverlogs limit 10").show()

+--------+-------------------+
|loglevel|            logtime|
+--------+-------------------+
|   DEBUG|2013-06-03 03:12:39|
|   DEBUG|2014-12-09 08:31:51|
|    INFO|2016-12-05 07:56:54|
|    INFO|2016-02-01 18:08:46|
|   DEBUG|2015-05-02 10:28:25|
|   DEBUG|2013-10-07 03:10:58|
|    INFO|2016-03-09 02:16:02|
|    INFO|2013-06-12 10:59:20|
|    WARN|2017-05-22 10:13:57|
|   ERROR|2015-01-15 12:27:40|
+--------+-------------------+



In [16]:
spark.sql("desc table serverlogs").show()

+--------+---------+-------+
|col_name|data_type|comment|
+--------+---------+-------+
|loglevel|   string|   null|
| logtime|timestamp|   null|
+--------+---------+-------+



### date_format ()

In [17]:
spark.sql("select loglevel, date_format(logtime,'MMMM') as month from serverlogs").show(10)

+--------+--------+
|loglevel|   month|
+--------+--------+
|    INFO|  August|
|    WARN| January|
|    INFO|    June|
|    INFO| January|
|   DEBUG|    July|
|    INFO|February|
|    INFO|    July|
|    INFO|   April|
|   DEBUG|November|
|    INFO|  August|
+--------+--------+
only showing top 10 rows



In [18]:
spark.sql("select loglevel, date_format(logtime,'M') as month from serverlogs").show(10)

+--------+-----+
|loglevel|month|
+--------+-----+
|    INFO|    8|
|    WARN|    1|
|    INFO|    6|
|    INFO|    1|
|   DEBUG|    7|
|    INFO|    2|
|    INFO|    7|
|    INFO|    4|
|   DEBUG|   11|
|    INFO|    8|
+--------+-----+
only showing top 10 rows



In [19]:
spark.sql("select loglevel, date_format(logtime,'MMM') as month from serverlogs").show(10)

+--------+-----+
|loglevel|month|
+--------+-----+
|    INFO|  Aug|
|    WARN|  Jan|
|    INFO|  Jun|
|    INFO|  Jan|
|   DEBUG|  Jul|
|    INFO|  Feb|
|    INFO|  Jul|
|    INFO|  Apr|
|   DEBUG|  Nov|
|    INFO|  Aug|
+--------+-----+
only showing top 10 rows



In [20]:
spark.sql("""
select loglevel, date_format(logtime,'MMMM') as month, count(*) as total 
from serverlogs group by loglevel,month order by loglevel""").show(10)

+--------+--------+-----+
|loglevel|   month|total|
+--------+--------+-----+
|   DEBUG|   April|41869|
|   DEBUG|November|33366|
|   DEBUG|   March|41652|
|   DEBUG| October|41936|
|   DEBUG| January|41961|
|   DEBUG|February|41734|
|   DEBUG|    June|41774|
|   DEBUG|    July|42085|
|   DEBUG|December|41749|
|   DEBUG|     May|41785|
+--------+--------+-----+
only showing top 10 rows



In [24]:
spark.sql("""
select loglevel, 
date_format(logtime,'MMMM') as month, 
date_format(logtime,'M')as month_num,  
count(*) as total 
from serverlogs group by loglevel,month,month_num order by month_num  
""").show(10)

+--------+-------+---------+-----+
|loglevel|  month|month_num|total|
+--------+-------+---------+-----+
|    WARN|January|        1| 8217|
|   DEBUG|January|        1|41961|
|    INFO|January|        1|29119|
|   ERROR|January|        1| 4054|
|   FATAL|January|        1|   94|
|   DEBUG|October|       10|41936|
|    WARN|October|       10| 8226|
|    INFO|October|       10|29018|
|   ERROR|October|       10| 4040|
|   FATAL|October|       10|   92|
+--------+-------+---------+-----+
only showing top 10 rows



In [23]:
spark.sql("""
select loglevel, 
date_format(logtime,'MMMM') as month, 
int(date_format(logtime,'M')) as month_num,  
count(*) as total 
from serverlogs group by loglevel,month,month_num order by month_num  
""").show(10)

+--------+--------+---------+-----+
|loglevel|   month|month_num|total|
+--------+--------+---------+-----+
|    INFO| January|        1|29119|
|    WARN| January|        1| 8217|
|   ERROR| January|        1| 4054|
|   FATAL| January|        1|   94|
|   DEBUG| January|        1|41961|
|   FATAL|February|        2|   72|
|    INFO|February|        2|28983|
|    WARN|February|        2| 8266|
|   ERROR|February|        2| 4013|
|   DEBUG|February|        2|41734|
+--------+--------+---------+-----+
only showing top 10 rows

